# 02-2. 규제 기반 Improved CNN 학습

이 노트북은 baseline과 OpenCV 증강 실험 이후의 개선 실험입니다.

핵심 목적은 **train 데이터에 과하게 맞는 overfitting을 줄이기 위해 규제 방법을 추가하는 것**입니다.

## Step 0. [디벨롭] 이번 노트북의 목표를 정리합니다

02 baseline과 02-1 OpenCV 실험에서 공통적으로 train loss는 빠르게 낮아졌지만 validation loss가 다시 증가하는 흐름을 확인했습니다.

따라서 이번 노트북에서는 데이터 수를 무작정 늘리는 것보다, 모델이 train 데이터에만 과하게 맞지 않도록 **규제 방법**을 추가합니다.

이번 실험의 기본 순서는 다음과 같습니다.

1. 원본 train 데이터로 규제 모델을 먼저 테스트합니다.
2. 결과가 괜찮으면 같은 규제 모델을 OpenCV 증강 train 데이터에도 적용합니다.
3. OpenCV 데이터에서도 과적합이 강하면 증강 이미지 수를 줄이는 방향으로 조정합니다.

## Step 1. [디벨롭] 이번 실험의 위치를 표로 정리합니다

| 실험 | 데이터 | 모델/학습 변화 | 확인할 것 |
|---|---|---|---|
| 02 baseline | 원본 train | SimpleCNN | 기준 성능과 overfitting 여부 |
| 02-1 OpenCV | 원본 + OpenCV 증강 train | SimpleCNN | 데이터 증강만으로 일반화가 좋아지는지 |
| 02-2 원본 규제 | 원본 train | BatchNorm + Dropout + weight decay + LR Scheduler | 규제가 overfitting을 줄이는지 |
| 02-2 OpenCV 규제 | 원본 + OpenCV 증강 train | 같은 규제 모델 | 증강과 규제를 함께 쓰면 좋아지는지 |

중요한 점은 OpenCV 증강 데이터부터 바로 다시 쓰지 않는 것입니다. 먼저 원본 데이터에서 규제 모델이 효과가 있는지 확인한 뒤, 그 다음 OpenCV 증강 데이터를 적용합니다.

## Step 2. 필요한 패키지와 경로를 준비합니다

기본 구조는 02와 비슷합니다. 다만 이번에는 규제 실험 결과를 따로 저장하기 위해 checkpoint와 summary 파일 이름을 분리합니다.

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
FIGURE_DIR = PROJECT_DIR / "outputs" / "figures"
METRICS_DIR = PROJECT_DIR / "outputs" / "metrics"
CHECKPOINT_DIR = PROJECT_DIR / "outputs" / "checkpoints"

for directory in [FIGURE_DIR, METRICS_DIR, CHECKPOINT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SPLIT_CSV_PATH = PROCESSED_DIR / "pet_binary_split_70_15_15_breed_stratified.csv"
AUGMENTED_TRAIN_CSV_PATH = PROCESSED_DIR / "pet_binary_split_70_15_15_opencv_augmented_train.csv"  # [디벨롭] 02-1에서 만든 증강 train CSV를 선택 실험에 사용합니다.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Step 3. split CSV와 선택적인 OpenCV 증강 CSV를 불러옵니다

원본 train/validation/test는 01에서 만든 split을 그대로 사용합니다.

OpenCV 증강 CSV는 02-1을 실행한 뒤에 생성됩니다. 아직 없으면 원본 규제 실험만 먼저 진행합니다.

In [ ]:
split_df = pd.read_csv(SPLIT_CSV_PATH)

train_df = split_df[split_df["split"] == "train"].reset_index(drop=True)
val_df = split_df[split_df["split"] == "validation"].reset_index(drop=True)
test_df = split_df[split_df["split"] == "test"].reset_index(drop=True)

if AUGMENTED_TRAIN_CSV_PATH.exists():
    augmented_train_df = pd.read_csv(AUGMENTED_TRAIN_CSV_PATH)  # [디벨롭] OpenCV 실험을 이어서 할 때 사용합니다.
else:
    augmented_train_df = None

pd.DataFrame({
    "dataset": ["original train", "validation", "test", "opencv augmented train csv exists"],
    "count": [len(train_df), len(val_df), len(test_df), augmented_train_df is not None],
})

## Step 4. Dataset과 DataLoader 준비 코드를 만듭니다

이미지 전처리는 02 baseline과 같은 128x128 기준을 유지합니다.

비교 실험에서는 전처리 조건을 유지해야 모델 변경 효과를 더 명확하게 볼 수 있습니다.

In [ ]:
IMAGE_SIZE = 128
BATCH_SIZE = 32
NUM_WORKERS = 0

LABEL_TO_INDEX = {"cat": 0, "dog": 1}

base_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])


def resolve_image_path(image_path):
    path = Path(image_path)
    if path.is_absolute():
        return path
    return PROJECT_DIR / path


class PetBinaryDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image = Image.open(resolve_image_path(row["image_path"])).convert("RGB")
        label = LABEL_TO_INDEX[row["label"]]

        if self.transform is not None:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


def make_loaders(train_dataframe):
    train_dataset = PetBinaryDataset(train_dataframe, transform=base_transform)
    val_dataset = PetBinaryDataset(val_df, transform=base_transform)
    test_dataset = PetBinaryDataset(test_df, transform=base_transform)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    return train_loader, val_loader, test_loader


original_train_loader, val_loader, test_loader = make_loaders(train_df)
len(original_train_loader), len(val_loader), len(test_loader)

## Step 5. [디벨롭] 규제 모델을 만듭니다

이번 모델은 SimpleCNN을 기준으로 다음 요소를 추가합니다.

- **BatchNorm:** 각 convolution layer 뒤에서 학습을 안정화합니다.
- **Dropout:** classifier에서 일부 뉴런을 임시로 꺼서 overfitting을 줄입니다.
- **작아진 Linear layer:** 기존보다 classifier 크기를 줄여 train 데이터를 외우는 힘을 낮춥니다.

모델 이름은 `RegularizedCNN`으로 둡니다.

In [ ]:
class RegularizedCNN(nn.Module):
    def __init__(self, dropout_p=0.5):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),  # [디벨롭] BatchNorm으로 feature 분포를 안정화합니다.
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),  # [디벨롭] 두 번째 convolution block에도 BatchNorm을 추가합니다.
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),  # [디벨롭] 깊은 feature에서도 학습이 흔들리지 않도록 합니다.
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=dropout_p),  # [디벨롭] train 데이터만 외우는 것을 줄이기 위해 일부 뉴런을 끕니다.
            nn.Linear(64 * 16 * 16, 64),  # [디벨롭] baseline의 hidden size 128보다 작게 만들어 모델 복잡도를 낮춥니다.
            nn.ReLU(),
            nn.Dropout(p=dropout_p),  # [디벨롭] classifier 뒤쪽에도 Dropout을 한 번 더 적용합니다.
            nn.Linear(64, 2),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = RegularizedCNN().to(device)
model

## Step 6. 학습과 평가 함수를 준비합니다

학습 루프의 핵심 흐름은 02와 같습니다.

`optimizer.zero_grad -> forward -> loss -> backward -> optimizer.step` 순서를 유지합니다.

In [ ]:
def train_one_epoch(model, data_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in data_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


def evaluate(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

## Step 7. [디벨롭] 규제 학습 설정을 함수로 묶습니다

이번에는 원본 train과 OpenCV train을 같은 조건으로 비교해야 하므로 학습 설정을 함수로 묶습니다.

새로 추가되는 규제 요소는 다음입니다.

- `weight_decay=1e-4`: 가중치가 너무 커지는 것을 억제합니다.
- `lr=0.0005`: baseline보다 learning rate를 낮춰 더 조심스럽게 학습합니다.
- `ReduceLROnPlateau`: validation loss가 잘 안 내려가면 learning rate를 줄입니다.

In [ ]:
def run_regularized_experiment(train_dataframe, experiment_name, checkpoint_name):
    train_loader, val_loader, _ = make_loaders(train_dataframe)

    model = RegularizedCNN(dropout_p=0.5).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        model.parameters(),
        lr=0.0005,          # [디벨롭] baseline보다 낮은 learning rate로 빠른 과적합을 줄여봅니다.
        weight_decay=1e-4,  # [디벨롭] L2 규제 역할을 해서 가중치가 너무 커지는 것을 억제합니다.
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2,
    )  # [디벨롭] validation loss가 정체되면 learning rate를 줄입니다.

    max_epochs = 30
    patience = 5
    min_delta = 0.0
    checkpoint_path = CHECKPOINT_DIR / checkpoint_name

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
        "learning_rate": [],
    }

    best_val_loss = float("inf")
    epochs_without_improvement = 0
    start_time = time.time()

    for epoch in range(max_epochs):
        train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)

        scheduler.step(val_loss)  # [디벨롭] validation loss를 보고 learning rate를 조정합니다.
        current_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)
        history["learning_rate"].append(current_lr)

        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            epochs_without_improvement = 0
            torch.save(
                {
                    "epoch": epoch + 1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_val_loss": best_val_loss,
                    "label_to_index": LABEL_TO_INDEX,
                    "image_size": IMAGE_SIZE,
                    "experiment_name": experiment_name,
                },
                checkpoint_path,
            )
            checkpoint_message = "best model saved"
        else:
            epochs_without_improvement += 1
            checkpoint_message = f"no improvement ({epochs_without_improvement}/{patience})"

        print(
            f"epoch {epoch + 1}/{max_epochs} | "
            f"train loss {train_loss:.4f}, train acc {train_accuracy:.4f} | "
            f"val loss {val_loss:.4f}, val acc {val_accuracy:.4f} | "
            f"lr {current_lr:.6f} | {checkpoint_message}"
        )

        if epochs_without_improvement >= patience:
            print("EarlyStopping 조건을 만족해서 학습을 멈춥니다.")
            break

    elapsed_time = time.time() - start_time
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])

    result = {
        "experiment_name": experiment_name,
        "history": history,
        "best_val_loss": best_val_loss,
        "best_val_accuracy": max(history["val_accuracy"]),
        "epochs_ran": len(history["train_loss"]),
        "elapsed_time_seconds": elapsed_time,
        "checkpoint_path": checkpoint_path,
    }
    return result

## Step 8. [디벨롭] 원본 train 데이터로 규제 모델을 먼저 학습합니다

먼저 원본 train 데이터만 사용합니다.

이 단계의 목적은 OpenCV 증강 데이터의 영향 없이, 규제 모델 자체가 overfitting을 줄이는지 확인하는 것입니다.

In [ ]:
original_regularized_result = run_regularized_experiment(
    train_dataframe=train_df,
    experiment_name="regularized_original_train",
    checkpoint_name="best_03_regularized_original_cnn.pt",
)

original_regularized_result

## Step 9. [디벨롭] 원본 규제 모델의 loss curve를 확인합니다

train loss와 validation loss의 간격이 baseline보다 줄었는지 확인합니다.

규제가 효과적이라면 train loss가 너무 빠르게 0에 가까워지는 현상이 완화되고, validation loss도 덜 튀는 방향을 기대할 수 있습니다.

In [ ]:
history = original_regularized_result["history"]
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, history["train_loss"], marker="o", label="train loss")
plt.plot(epochs, history["val_loss"], marker="o", label="validation loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Regularized CNN Original Train/Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

original_regularized_loss_curve_path = FIGURE_DIR / "regularized_original_loss_curve.png"
plt.savefig(original_regularized_loss_curve_path, dpi=150)
plt.show()

print(original_regularized_loss_curve_path)

## Step 10. [디벨롭] OpenCV 증강 train 데이터로 같은 규제 모델을 선택 실험합니다

원본 train에서 규제 모델이 괜찮다면, 그 다음 같은 모델을 OpenCV 증강 train 데이터에도 적용합니다.

기본값은 `False`입니다. 원본 규제 실험 결과를 먼저 확인한 뒤 `True`로 바꿔 실행합니다.

In [ ]:
RUN_OPENCV_IMPROVED = False  # [디벨롭] 원본 규제 실험을 먼저 본 뒤 True로 바꿔 OpenCV 규제 실험을 실행합니다.

if RUN_OPENCV_IMPROVED:
    if augmented_train_df is None:
        raise FileNotFoundError("OpenCV 증강 train CSV가 없습니다. 먼저 02-1 Step 8~9를 실행해야 합니다.")

    opencv_regularized_result = run_regularized_experiment(
        train_dataframe=augmented_train_df,
        experiment_name="regularized_opencv_augmented_train",
        checkpoint_name="best_06_final_opencv_regularized_cnn.pt",
    )
else:
    opencv_regularized_result = None
    print("OpenCV 규제 실험은 아직 실행하지 않습니다. 필요하면 RUN_OPENCV_IMPROVED를 True로 바꿉니다.")

## Step 10-1. [디벨롭] 규제 실험 summary를 같은 파일에 저장합니다

02 baseline과 02-1 OpenCV 실험처럼, 02-2 결과도 같은 `outputs/metrics/all_training_summaries.csv`에 저장합니다.

이렇게 해야 나중에 여러 모델 결과를 한 번에 비교할 수 있습니다.

In [ ]:
def summarize_regularized_result(result, loss_curve_path=None):
    history = result["history"]
    best_val_loss = min(history["val_loss"])
    best_epoch = history["val_loss"].index(best_val_loss) + 1

    return {
        "experiment_name": result["experiment_name"],
        "split_csv": str(SPLIT_CSV_PATH.relative_to(PROJECT_DIR)),
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "epochs_ran": result["epochs_ran"],
        "best_epoch": best_epoch,
        "device": str(device),
        "elapsed_time_seconds": result["elapsed_time_seconds"],
        "final_train_loss": history["train_loss"][-1],
        "final_val_loss": history["val_loss"][-1],
        "final_train_accuracy": history["train_accuracy"][-1],
        "final_val_accuracy": history["val_accuracy"][-1],
        "best_val_loss": best_val_loss,
        "best_val_accuracy": history["val_accuracy"][best_epoch - 1],
        "loss_curve_path": str(loss_curve_path.relative_to(PROJECT_DIR)) if loss_curve_path is not None else "",
        "checkpoint_path": str(result["checkpoint_path"].relative_to(PROJECT_DIR)),
        "learning_rate_start": result["history"]["learning_rate"][0],
        "learning_rate_end": result["history"]["learning_rate"][-1],
    }


summary_records = [
    summarize_regularized_result(
        original_regularized_result,
        loss_curve_path=original_regularized_loss_curve_path,
    )
]

if opencv_regularized_result is not None:
    summary_records.append(summarize_regularized_result(opencv_regularized_result))

regularized_training_summary = pd.DataFrame(summary_records)

regularized_summary_path = METRICS_DIR / "regularized_training_summary.csv"
all_training_summary_path = METRICS_DIR / "all_training_summaries.csv"

regularized_training_summary.to_csv(regularized_summary_path, index=False)

if all_training_summary_path.exists():
    previous_summary = pd.read_csv(all_training_summary_path)
    previous_summary = previous_summary[~previous_summary["experiment_name"].isin(regularized_training_summary["experiment_name"])]
    all_training_summary = pd.concat([previous_summary, regularized_training_summary], ignore_index=True)
else:
    all_training_summary = regularized_training_summary.copy()

all_training_summary.to_csv(all_training_summary_path, index=False)

print(regularized_summary_path)
print(all_training_summary_path)
regularized_training_summary

## Step 11. [디벨롭] OpenCV에서도 과적합이 반복될 경우 증강 수를 줄이는 계획을 만듭니다

02-1에서는 원본 이미지 1장당 여러 증강 이미지가 생겼습니다. 데이터 수는 늘었지만, 실제로는 같은 원본 이미지의 변형이 많아서 모델이 그것까지 외웠을 가능성이 있습니다.

만약 OpenCV + 규제 모델에서도 validation loss가 계속 증가한다면, 증강 종류를 줄인 train CSV를 만들어 다시 실험합니다.

In [ ]:
REDUCED_AUGMENTATION_TYPES = ["original", "bright", "rotate", "flip"]  # [디벨롭] blur/contrast를 일단 제외해서 증강 수를 줄이는 후보입니다.

if augmented_train_df is not None and "augmentation_type" in augmented_train_df.columns:
    reduced_augmented_train_df = augmented_train_df[
        augmented_train_df["augmentation_type"].isin(REDUCED_AUGMENTATION_TYPES)
    ].reset_index(drop=True)

    reduced_augmented_train_csv_path = PROCESSED_DIR / "pet_binary_split_70_15_15_opencv_reduced_augmented_train.csv"
    reduced_augmented_train_df.to_csv(reduced_augmented_train_csv_path, index=False)

    display(pd.DataFrame({
        "dataset": ["full opencv augmented train", "reduced opencv augmented train"],
        "count": [len(augmented_train_df), len(reduced_augmented_train_df)],
    }))
    print(reduced_augmented_train_csv_path)
else:
    reduced_augmented_train_df = None
    print("아직 OpenCV 증강 CSV가 없거나 augmentation_type 컬럼이 없습니다.")

## Step 12. [디벨롭] 이번 실험의 해석 기준을 정리합니다

이번 노트북에서 확인할 질문은 다음과 같습니다.

- 원본 train + 규제 모델에서 validation loss가 baseline보다 안정적인가?
- train loss가 너무 빠르게 0에 가까워지는 현상이 줄었는가?
- OpenCV + 규제 모델에서도 과적합이 반복되는가?
- OpenCV 과적합이 반복된다면 증강 수를 줄였을 때 더 안정적인가?

발표에서는 다음처럼 말할 수 있습니다.

> OpenCV 증강만으로는 overfitting이 충분히 줄어들지 않았습니다. 그래서 다음 실험에서는 모델 쪽 규제 방법을 추가했습니다. 먼저 원본 train 데이터에서 BatchNorm, Dropout, weight decay, LR Scheduler가 validation loss 안정화에 도움이 되는지 확인하고, 이후 같은 규제 모델을 OpenCV 증강 데이터에도 적용했습니다. 만약 OpenCV 데이터에서도 과적합이 반복된다면 증강 이미지 수를 줄이는 방향으로 추가 실험을 진행합니다.